## Aim: to identify the similarity between the macrostates and attractors of the models and the number of attractors 

In [15]:
from pathlib import Path

import mpbn
import pandas as pd
import numpy as np

# PARAMETERS

models_folder = Path("/home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Boolean_model_BoNesis/Bonesis_test/ensemble")
macrostate_file = "/home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Boolean_model_BoNesis/Bonesis_test/data_binarized_tf_gene_sep.csv"
threshold = 0.95

# LOAD MACROSTATES
data = pd.read_csv(macrostate_file, index_col=0)

selected_models = []
summary = []

# LOOP OVER MODELS

for model_file in sorted(models_folder.glob("*.bnet")):

    print(f"\nProcessing {model_file.name}")

    # Compute attractors

    mbn = mpbn.MPBooleanNetwork(str(model_file))
    attractors = list(mbn.attractors())

    attractors = pd.DataFrame(attractors)

    if attractors.empty:
        print("No attractor found")
        continue

    attractors = attractors[sorted(attractors.columns)]

    attractors.index = [
        f"Attractor{i+1}"
        for i in range(len(attractors))
    ]

    # Keep common genes

    common = attractors.columns.intersection(data.columns)

    A = attractors[common]
    D = data[common]

    # Similarity matrix

    similarity = pd.DataFrame(
        index=A.index,
        columns=D.index,
        dtype=float,
    )

    for attr_name, attr_row in A.iterrows():

        for state_name, state_row in D.iterrows():

            mask = ~pd.isna(state_row.values)

            similarity.loc[attr_name, state_name] = np.mean(
                attr_row.values[mask] ==
                state_row.values[mask]
            )

    
    # Best similarity

    # Attractors compatible with each macrostate
    S2_candidates = similarity.index[similarity["S2"] >= threshold].tolist()
    S3_candidates = similarity.index[similarity["S3"] >= threshold].tolist()

    keep = False

    for a2 in S2_candidates:
        for a3 in S3_candidates:
            if a2 != a3:
                keep = True
                selected_S2 = a2
                selected_S3 = a3
                break
        if keep:
            break

    best_S2 = similarity["S2"].max()
    best_S3 = similarity["S3"].max()

    summary.append({
        "model": model_file.name,
        "n_attractors": len(attractors),
        "best_S2": similarity["S2"].max(),
        "best_S3": similarity["S3"].max(),
        "S2_attractor": selected_S2 if keep else None,
        "S3_attractor": selected_S3 if keep else None,
        "selected": keep,
    })

    if keep:
        selected_models.append(model_file.name)

# RESULTS
summary = pd.DataFrame(summary)

print()
print("=" * 40)
print(f"Models analysed : {len(summary)}")
print(f"Selected models : {summary['selected'].sum()}")
print("=" * 40)
print()
#summary.to_csv("summary_models.csv", index=False)


Processing model_0.bnet

Processing model_1.bnet

Processing model_10.bnet

Processing model_11.bnet

Processing model_12.bnet

Processing model_13.bnet

Processing model_14.bnet

Processing model_15.bnet

Processing model_16.bnet

Processing model_17.bnet

Processing model_18.bnet

Processing model_19.bnet

Processing model_2.bnet

Processing model_20.bnet

Processing model_21.bnet

Processing model_22.bnet

Processing model_23.bnet

Processing model_24.bnet

Processing model_25.bnet

Processing model_26.bnet

Processing model_27.bnet

Processing model_28.bnet

Processing model_29.bnet

Processing model_3.bnet

Processing model_30.bnet

Processing model_31.bnet

Processing model_32.bnet

Processing model_33.bnet

Processing model_34.bnet

Processing model_35.bnet

Processing model_36.bnet

Processing model_37.bnet

Processing model_38.bnet

Processing model_39.bnet

Processing model_4.bnet

Processing model_40.bnet

Processing model_41.bnet

Processing model_42.bnet

Processing model

In [16]:
print(summary)

            model  n_attractors  best_S2  best_S3 S2_attractor S3_attractor  \
0    model_0.bnet             2      1.0      1.0   Attractor1   Attractor2   
1    model_1.bnet             2      1.0      1.0   Attractor1   Attractor2   
2   model_10.bnet             2      1.0      1.0   Attractor1   Attractor2   
3   model_11.bnet             2      1.0      1.0   Attractor1   Attractor2   
4   model_12.bnet             2      1.0      1.0   Attractor1   Attractor2   
..            ...           ...      ...      ...          ...          ...   
95  model_95.bnet             2      1.0      1.0   Attractor1   Attractor2   
96  model_96.bnet             2      1.0      1.0   Attractor1   Attractor2   
97  model_97.bnet             2      1.0      1.0   Attractor1   Attractor2   
98  model_98.bnet             2      1.0      1.0   Attractor1   Attractor2   
99  model_99.bnet             2      1.0      1.0   Attractor1   Attractor2   

    selected  
0       True  
1       True  
2     

In [17]:
print(similarity)

macrostate        S0       S11        S2        S3
Attractor1  0.104377  0.222689  1.000000  0.722022
Attractor2  0.060606  0.117647  0.732394  1.000000
